# Clasificar clubes externos (juvenil / padre) recorriendo las carreras

**Idea:** por cada jugador se camina su carrera en orden. Se frena en el primer club "padre" (primer equipo), salvo para los clubes en los que el jugador empezó antes de los 24 años (ver abajo).
Los clubes que no conocemos y aparecen antes de ese punto se visitan en Transfermarkt.
De cada página se leen **todos** los equipos de la lista del encabezado: el primero es el padre, el resto son sus juveniles.

**Lee solo `silver_transfers` y `bronze_youth_teams_v2` (y su propia tabla de salida). No usa ninguna tabla gold**, así que corre después de silver y antes de gold.

**Tabla de salida:** `workspace.inferiores.bronze_clubes_externos`

| Columna | Contenido |
|---|---|
| `club_id` | ID del club en Transfermarkt |
| `nombre` | Nombre tomado de la lista de la página o, si no hay lista, de `silver_transfers` |
| `rol` | `padre`, `juvenil`, `no_club`, `sin_lista` o `sin_pagina` |
| `parent_club_id` | ID del club padre, solo para los `juvenil` |
| `fuente` | `marcador`, `visita` (se visitó la página de ese club) o `lista_de_otra_pagina` (apareció en la lista de otro club) |
| `visitado_en` | Momento en que se generó la fila |

Roles: `padre`, `juvenil`, `no_club` (marcadores de TM: Without Club, Unknown, Retired), `sin_lista` (la página no trae lista de equipos), `sin_pagina` (404).
`sin_lista` y `sin_pagina` cortan siempre el recorrido del jugador. `padre` lo corta solo para los clubes en los que el jugador empezó a los 24 años o más; los clubes en los que empezó antes de los 24 se siguen clasificando aunque estén después de su primer club padre.

**Orden de ejecución:** celda 1 (funciones), celda 2 (control), celda 3 (rondas, se puede cortar y retomar), celda 4 (resumen).
Si se reinicia la sesión, correr la celda 1 de nuevo y seguir.

## Reglas de acceso al sitio que aplica el código

- Descarga `robots.txt` de `www.transfermarkt.com` al inicio y, antes de cada pedido no cacheado, consulta si la ruta está permitida. Si no lo está, lanza `RuntimeError`.
- Espera 5 s (`DELAY`) antes de cada pedido no cacheado.
- Ante HTTP `202`, `403` o `429` espera 120 s y reintenta, hasta 2 veces. Si vuelve a pasar, lanza `RuntimeError`. La celda de rondas atrapa ese error, guarda lo pendiente, imprime `FRENO` y termina sin error.
- Guarda cada página descargada en un Volume de Databricks y la lee de ahí en las corridas siguientes.

## Control del ciclo
`%run ./ciclo` carga las funciones del notebook `ciclo`. La celda siguiente llama a `motivo_no_corre("publicar")`: si el ciclo no está listo para publicar (todavía faltan jugadores o ya se publicó), termina con `SALTEADO: <motivo>` y no hace nada.

In [ ]:
%run ./ciclo

In [ ]:
motivo_salida = motivo_no_corre("publicar")
if motivo_salida:
    dbutils.notebook.exit("SALTEADO: " + motivo_salida)

## Celda 1: configuración y funciones
Correr primero en cada sesión.

| Elemento | Qué hace |
|---|---|
| `TM`, `UA`, `CACHE`, `DELAY` | Sitio, User-Agent, carpeta de caché en el Volume y espera entre páginas (5 s) |
| `TABLA`, `COLS`, `SCHEMA` | Tabla de salida y su esquema (todas las columnas son texto). La celda crea la tabla si no existe |
| `curl(url, out)` y `get(url)` | Descarga con caché, `robots.txt`, espera, reintentos (hasta 6 intentos) y manejo de freno |
| `bloque(page, cid)` | Devuelve el primer `<ul>` con `tiny_wappen` que contiene el enlace del propio club |
| `leer_lista(page, cid)` | Lista de `(nombre, id)` de los equipos del encabezado, sin repetidos y en el orden de la página |
| `ahora()` | Hora actual en formato ISO (igual que la de `ciclo`) |
| `guardar(rows)` | Agrega filas a la tabla de salida |
| `FAMILIA_SQL` y `FAMILIA` | Familia ya conocida: los IDs de los equipos juveniles de `bronze_youth_teams_v2` (rol `juvenil`) y los de sus clubes (rol `padre`). Esos IDs no se vuelven a guardar ni visitar |
| `NOMBRES` | Nombre de cada club según `silver_transfers`, para las filas sin lista |
| `MARCADORES` | Seis IDs de Transfermarkt que no son clubes (Without Club, Unknown, Retired, Career break, Disqualification y Own Youth). Se guardan como `no_club` con `fuente = marcador` si no estaban |
| `pendientes()` | Devuelve, por club, la cantidad de jugadores para los que ese club es el primero sin clasificar de su carrera, ordenado de mayor a menor |

**Cómo decide `pendientes()`**

1. Arma la carrera de cada jugador con los clubes de sus traspasos, en orden cronológico y sin los traspasos futuros (`upcoming`).
2. A cada club le asigna su rol: el de la familia conocida o el de `bronze_clubes_externos`. Sin rol, el club está sin clasificar.
3. Para cada jugador calcula dos cortes: la posición del primer club con rol `padre`, `sin_lista` o `sin_pagina`, y la del primer club con rol `sin_lista` o `sin_pagina`.
4. Un club sin clasificar entra si está antes del primer corte, o si el jugador llegó a ese club con menos de 24 años (o sin edad registrada) y el club está antes del segundo corte.
5. De los que entran, se toma el más temprano de cada jugador.

Al final imprime cuántos clubes y jugadores quedan pendientes.

In [ ]:
# CELDA 1: configuracion y funciones (correr primero en cada sesion)
import subprocess, re, time, hashlib, os, urllib.robotparser
from datetime import datetime
from html import unescape
from pyspark.sql.types import StructType, StructField, StringType

TM = "https://www.transfermarkt.com"
UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36"
CACHE = "/Volumes/workspace/inferiores/scraper/cache"
DELAY = 5.0   # segundos entre paginas (bajar a 3 solo si TM no frena)
TABLA = "workspace.inferiores.bronze_clubes_externos"
COLS = ["club_id", "nombre", "rol", "parent_club_id", "fuente", "visitado_en"]
SCHEMA = StructType([StructField(c, StringType()) for c in COLS])


def curl(url, out):
    r = subprocess.run(["curl", "-sL", "--max-time", "40", "-A", UA, "-w", "%{http_code}", "-o", out, url],
                       capture_output=True, text=True, timeout=60)
    return r.stdout.strip()


curl(TM + "/robots.txt", "/tmp/robots.txt")
rp = urllib.robotparser.RobotFileParser()
rp.parse(open("/tmp/robots.txt", encoding="utf-8", errors="replace").read().splitlines())


def get(url):
    p = f"{CACHE}/{hashlib.md5(url.encode()).hexdigest()}.html"
    if os.path.exists(p):
        return "200", open(p, encoding="utf-8", errors="replace").read()
    if not rp.can_fetch(UA, url):
        raise RuntimeError(f"robots.txt prohibe: {url}")
    time.sleep(DELAY)
    tmp = p + ".tmp"
    code = ""
    freno = 0
    for attempt in range(6):
        code = curl(url, tmp)
        if code == "200":
            os.replace(tmp, p)
            return code, open(p, encoding="utf-8", errors="replace").read()
        if code in ("202", "403", "429"):
            freno += 1
            if freno > 2:
                raise RuntimeError(f"HTTP {code} en {url}: freno")
            print(f"  HTTP {code}: espero 120 s y reintento ({freno}/2)")
            time.sleep(120)
            continue
        if code == "404":
            return code, ""
        time.sleep(5 * (attempt + 1))
    return code or "0", ""


ITEM = re.compile(r'<a[^>]*title="([^"]*)"[^>]*href="/([^/"]+)/startseite/verein/(\d+)"')


def bloque(page, cid):
    for m in re.finditer(r"(?s)<ul[^>]*>.*?</ul>", page):
        b = m.group(0)
        if "tiny_wappen" in b and f'/verein/{cid}"' in b:
            return b
    return ""


def leer_lista(page, cid):
    """Lista [(nombre, id)] de equipos del encabezado, sin repetidos y en el orden de la pagina."""
    vistos, items = set(), []
    for n, s, i in ITEM.findall(bloque(page, cid)):
        if i not in vistos:
            vistos.add(i)
            items.append((unescape(n), i))
    return items


def ahora():
    return datetime.now().isoformat(timespec="seconds")


spark.sql(f"""CREATE TABLE IF NOT EXISTS {TABLA}
              (club_id STRING, nombre STRING, rol STRING, parent_club_id STRING,
               fuente STRING, visitado_en STRING)""")


def guardar(rows):
    if rows:
        spark.createDataFrame(rows, SCHEMA).write.mode("append").saveAsTable(TABLA)


# Familia ya conocida = los 105 padres + sus 215 equipos juveniles, tal como salen de bronze_youth_teams_v2
# (la misma fuente que usa gold_club_familia; aca no se lee ninguna tabla gold)
FAMILIA_SQL = """
    SELECT CAST(team_id AS STRING) AS club_id, 'juvenil' AS rol
    FROM workspace.inferiores.bronze_youth_teams_v2
    UNION ALL
    SELECT DISTINCT CAST(club_id AS STRING) AS club_id, 'padre' AS rol
    FROM workspace.inferiores.bronze_youth_teams_v2
"""
# ids de la familia: nunca se vuelven a guardar ni visitar
FAMILIA = {r.club_id for r in spark.sql(FAMILIA_SQL).collect()}

# nombres de club tomados de silver_transfers (para los casos sin lista)
NOMBRES = {r.club_id: r.nombre for r in spark.sql("""
    WITH n AS (
      SELECT CAST(from_club_id AS STRING) AS club_id, from_club_name AS nombre FROM workspace.inferiores.silver_transfers
      UNION ALL
      SELECT CAST(to_club_id AS STRING), to_club_name FROM workspace.inferiores.silver_transfers
    )
    SELECT club_id, MAX(nombre) AS nombre FROM n GROUP BY club_id""").collect()}

# marcadores de Transfermarkt que no son clubes: se saltan en el recorrido
ya = {r.club_id for r in spark.table(TABLA).select("club_id").collect()}
MARCADORES = [("515", "Without Club"), ("75", "Unknown"), ("123", "Retired"),
              ("2113", "Career break"), ("2077", "Disqualification"),
              ("12604", "Own Youth")]  # Own Youth: ojo, puede servir para el reparto de formacion en la gold
guardar([(i, n, "no_club", None, "marcador", ahora()) for i, n in MARCADORES if i not in ya])


def pendientes():
    """Por cada jugador: primer club sin clasificar entre los que empezó antes de los 24 años y antes de su
    primer corte (sin_lista/sin_pagina). Ya no corta en el primer padre.
    Devuelve club_id, jugadores (distintos), ordenado por frecuencia."""
    return spark.sql(f"""
    WITH t AS (
      SELECT player_id,
             CAST(from_club_id AS STRING) AS from_id,
             CAST(to_club_id   AS STRING) AS to_id,
             age,
             ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY orden DESC) AS k,
             LAG(age)     OVER (PARTITION BY player_id ORDER BY orden DESC) AS age_prev
      FROM workspace.inferiores.silver_transfers
      WHERE COALESCE(upcoming, false) = false
    ),
    seq AS (
      SELECT player_id, k - 0.5 AS pos, from_id AS club_id, age_prev AS edad_inicio FROM t
      UNION ALL
      SELECT player_id, k AS pos, to_id AS club_id, age AS edad_inicio FROM t
    ),
    r AS (
      {FAMILIA_SQL}
      UNION ALL
      SELECT club_id, rol FROM {TABLA}
    ),
    carrera AS (
      SELECT s.player_id, s.pos, s.club_id, s.edad_inicio, r.rol
      FROM seq s LEFT JOIN r ON s.club_id = r.club_id
      WHERE s.club_id IS NOT NULL
    ),
    corte AS (
      SELECT player_id,
             MIN(CASE WHEN rol IN ('padre', 'sin_lista', 'sin_pagina') THEN pos END) AS pos_padre,
             MIN(CASE WHEN rol IN ('sin_lista', 'sin_pagina') THEN pos END)           AS pos_duro
      FROM carrera GROUP BY player_id
    ),
    fu AS (
      SELECT c.player_id, MIN(c.pos) AS pos
      FROM carrera c LEFT JOIN corte x ON x.player_id = c.player_id
      WHERE c.rol IS NULL
        AND ( x.pos_padre IS NULL OR c.pos < x.pos_padre
              OR ( (c.edad_inicio IS NULL OR c.edad_inicio < 24)
                   AND (x.pos_duro IS NULL OR c.pos < x.pos_duro) ) )
      GROUP BY c.player_id
    ),
    p AS (
      SELECT c.player_id, c.club_id
      FROM carrera c JOIN fu ON fu.player_id = c.player_id AND fu.pos = c.pos
    )
    SELECT club_id, COUNT(DISTINCT player_id) AS jugadores
    FROM p GROUP BY club_id ORDER BY jugadores DESC, club_id
    """)


p0 = pendientes().collect()
print(f"Listo. Pendientes ahora: {len(p0)} clubes, {sum(r.jugadores for r in p0)} jugadores.")

## Celda 2: control
Verifica la regla "el primero de la lista es el padre" con 30 clubes que ya se conocen: 10 padres y 20 juveniles de `bronze_youth_teams_v2`, elegidos de forma determinista con `md5(club_id)`. Para cada uno lee la lista de su página y comprueba que el primero sea el propio club si es padre, y otro club si es juvenil.

Imprime `CONTROL: <ok>/30 OK` y las fallas. Se espera 30/30. La celda solo imprime el resultado: no detiene la celda 3, así que si falla hay que parar a mano.

In [ ]:
# CELDA 2: CONTROL. Verifica la regla (el primero de la lista es el padre) con clubes que YA conocemos
# (salen de bronze_youth_teams_v2). Se espera 30/30. Si falla, NO correr la celda 3.
spark.sql(FAMILIA_SQL).createOrReplaceTempView("familia_conocida")
display(spark.sql("SELECT rol, COUNT(*) AS n FROM familia_conocida GROUP BY rol"))

padres = spark.sql("""SELECT club_id, rol FROM familia_conocida
                      WHERE rol = 'padre' ORDER BY md5(club_id) LIMIT 10""").collect()
juvs = spark.sql("""SELECT club_id, rol FROM familia_conocida
                    WHERE rol = 'juvenil' ORDER BY md5(club_id) LIMIT 20""").collect()

ok, mal = 0, []
for r in padres + juvs:
    code, page = get(f"{TM}/x/startseite/verein/{r.club_id}")
    items = leer_lista(page, r.club_id)
    if not items:
        mal.append((r.club_id, r.rol, f"sin lista (HTTP {code})"))
        continue
    esperado_padre = (r.rol == "padre")
    obtenido_padre = (items[0][1] == r.club_id)
    if esperado_padre == obtenido_padre:
        ok += 1
    else:
        mal.append((r.club_id, r.rol, f"primero de la lista: {items[0]}"))

print(f"CONTROL: {ok}/{len(padres) + len(juvs)} OK")
for m in mal:
    print("  FALLA:", m)

## Celda 3: rondas
Se puede cortar y volver a correr: retoma donde quedó.

| Parámetro | Valor | Para qué sirve |
|---|---|---|
| `MAX_VISITAS` | `1000` | Visitas nuevas por corrida |
| `MAX_MIN` | `500` | Minutos máximos por corrida |

**Cada ronda**

1. Calcula `pendientes()` y descarta los clubes que fallaron en esta corrida. Si no queda ninguno, termina.
2. Visita cada club pendiente (salvo los ya conocidos o de la familia) y guarda según el resultado:

| Resultado de la visita | Qué se guarda |
|---|---|
| HTTP 404 | El club con rol `sin_pagina` |
| HTTP distinto de 200 | Nada: el club se reintenta en la próxima corrida |
| Página sin lista de equipos | El club con rol `sin_lista` |
| Página con lista | Todos los equipos de la lista: el primero como `padre` y el resto como `juvenil` con su `parent_club_id`. `fuente` es `visita` para el club visitado y `lista_de_otra_pagina` para los demás |

3. Escribe cada 25 visitas y al final de la ronda. Después vuelve a calcular los pendientes, porque lo recién clasificado cambia el recorrido de los jugadores.
4. Si una ronda no hizo ninguna visita nueva, corta para no entrar en un bucle.

La celda termina cuando no quedan pendientes, cuando se alcanza un límite (`MAX_VISITAS` o `MAX_MIN`) o cuando hay un freno. En los tres casos escribe lo que quedaba en el búfer.

In [ ]:
# CELDA 3: RONDAS. Se puede cortar (MAX_VISITAS / MAX_MIN) y volver a correr: retoma donde quedo.
MAX_VISITAS = 1000     # visitas nuevas por corrida
MAX_MIN = 500          # minutos maximos por corrida


class Parar(Exception):
    pass


buffer, conocidos, fallidos = [], set(), set()
visitas, t0 = 0, time.time()


def flush():
    guardar(list(buffer))
    buffer.clear()


try:
    while True:
        pend = [(r.club_id, r.jugadores) for r in pendientes().collect()]
        pend = [(c, j) for c, j in pend if c not in fallidos]
        if not pend:
            print("No quedan clubes pendientes.")
            break
        print(f"--- ronda: {len(pend)} clubes pendientes ---")
        visitas_ronda = 0
        for cid, nj in pend:
            if cid in conocidos or cid in FAMILIA:
                continue
            if visitas >= MAX_VISITAS or (time.time() - t0) / 60 > MAX_MIN:
                raise Parar()
            code, page = get(f"{TM}/x/startseite/verein/{cid}")
            visitas += 1
            visitas_ronda += 1
            if code == "404":
                buffer.append((cid, NOMBRES.get(cid), "sin_pagina", None, "visita", ahora()))
                conocidos.add(cid)
            elif code != "200":
                fallidos.add(cid)
                print(f"  HTTP {code} en {cid}: se reintenta en la proxima corrida")
            else:
                items = leer_lista(page, cid)
                if not items:
                    buffer.append((cid, NOMBRES.get(cid), "sin_lista", None, "visita", ahora()))
                    conocidos.add(cid)
                else:
                    padre_id = items[0][1]
                    for nombre, i in items:
                        if i in conocidos or i in FAMILIA:
                            continue
                        rol = "padre" if i == padre_id else "juvenil"
                        parent = None if i == padre_id else padre_id
                        fuente = "visita" if i == cid else "lista_de_otra_pagina"
                        buffer.append((i, nombre, rol, parent, fuente, ahora()))
                        conocidos.add(i)
            if visitas % 25 == 0:
                flush()
                print(f"  {visitas} visitas | {(time.time() - t0) / 60:.1f} min")
        flush()
        if visitas_ronda == 0:
            print("Ronda sin visitas nuevas: corto para no entrar en un bucle.")
            break
except Parar:
    print(f"Corte por limite: {visitas} visitas, {(time.time() - t0) / 60:.1f} min. Volver a correr esta celda para seguir.")
except RuntimeError as e:
    print("FRENO:", e)
finally:
    flush()

## Celda 4: resumen
Muestra la cantidad de clubes por `rol` y `fuente`, imprime cuántos clubes y jugadores siguen pendientes (con 0 pendientes la clasificación está completa) y lista los clubes con rol `sin_lista` o `sin_pagina`, que conviene revisar a mano.

In [ ]:
# CELDA 4: RESUMEN
display(spark.sql(f"SELECT rol, fuente, COUNT(*) AS clubes FROM {TABLA} GROUP BY rol, fuente ORDER BY rol, fuente"))

pend = pendientes().collect()
print(f"Pendientes: {len(pend)} clubes, {sum(r.jugadores for r in pend)} jugadores (con 0 pendientes la clasificacion esta completa).")

print("Paginas sin lista o sin pagina (revisar a mano):")
display(spark.sql(f"SELECT club_id, nombre, rol FROM {TABLA} WHERE rol IN ('sin_lista', 'sin_pagina') ORDER BY rol, nombre"))